<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Compare subsidy and distortion metrics across scenarios for one subsidies-distortions run.
- **Primary inputs**: `runs/subsidies_distortions/<run_name>/*/subsidies_distortion.csv` and `runs/subsidies_distortions/<run_name>/*/output.csv`.
- **Primary outputs**: `img/subsidies_<select>.csv`, bar charts, `img/subsidies_gap.png`, and per-scenario subsidy-vs-distortion scatters.
- **Refactor role**: Consolidate exploratory cells into a reproducible workflow with explicit configuration and reusable helpers.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Resolve a subsidies-distortions run folder and load all scenario tables.
2. Build `Subsidies Gap = Distortion - Subsidies` for each scenario.
3. Extract selected subsidy indicators from `output.csv`.
4. Export a summary CSV and generate standardized plots.

### Inputs
- `project/analysis/post_processing/policy_assessment/runs/subsidies_distortions/<run_name>/*/subsidies_distortion.csv`
- `project/analysis/post_processing/policy_assessment/runs/subsidies_distortions/<run_name>/*/output.csv`

### Outputs
- `project/analysis/post_processing/policy_assessment/runs/subsidies_distortions/<run_name>/img/subsidies_<select>.csv`
- `project/analysis/post_processing/policy_assessment/runs/subsidies_distortions/<run_name>/img/subsidies_gap.png`
- `project/analysis/post_processing/policy_assessment/runs/subsidies_distortions/<run_name>/img/subsidies_distortion_<scenario>.png`

### How To Reuse
- Update `run_name` and optional filters in the configuration cell.


# Analyze subsidy-distortion runs


In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sys
sys.path.insert(0, "../../../..")

from project.analysis.post_processing.shared.io_runs import load_output_bundle
from project.analysis.post_processing.shared.paths import resolve_run
from project.utils import make_scatter_plot


In [ ]:
# Run selection
run_name = "test"
selected_scenarios = None  # Example: ["subsidies_2018", "subsidies_2021", "subsidies_2024"]

# Indicator extraction
select = "avg"  # "avg" or a year available in output.csv, e.g. 2019
indicator_pattern = r"^Subsidies total (.*) \(Million euro\)$"
reference_pattern = None  # Example: r"^Investment total (.*) \(Billion euro\)$"
reference_scale = 1e3

# Plot controls
same_ylim = True
fontsize = 14

remove_technologies = [
    "Electricity-Direct electric",
    "Natural gas-Performance boiler",
    "Oil fuel-Performance boiler",
    "Wood fuel-Performance boiler",
    "Heating-District heating",
]


In [ ]:
def load_child_csv_bundle(folder, filename, index_col=None):
    """Load one CSV file from each direct child folder."""
    result = {}
    for child in sorted(folder.iterdir()):
        csv_path = child / filename
        if child.is_dir() and csv_path.is_file():
            result[child.name] = pd.read_csv(csv_path, index_col=index_col)
    return result


def keep_scenarios(bundle, scenarios):
    """Filter bundle keys while preserving requested order."""
    if not scenarios:
        return bundle
    return {scenario: bundle[scenario] for scenario in scenarios if scenario in bundle}


def add_subsidies_gap(bundle):
    """Add a `Subsidies Gap` column to each table."""
    result = {}
    required = {"Subsidies", "Distortion"}

    for scenario, df in bundle.items():
        missing = required.difference(df.columns)
        if missing:
            raise KeyError("Missing columns for {}: {}".format(scenario, sorted(missing)))

        enriched = df.copy()
        enriched["Subsidies Gap"] = enriched["Distortion"] - enriched["Subsidies"]
        result[scenario] = enriched

    return result


def extract_indicator_values(
    table,
    pattern,
    select_value,
    reference=None,
    reference_multiplier=1e3,
):
    """Extract indicator rows matching a regex and return one series per metric."""
    index = table.index.to_series().astype(str)
    mask = index.str.match(pattern)
    filtered = table.loc[mask].copy()

    if filtered.empty:
        return pd.Series(dtype=float)

    filtered.index = index[mask].str.extract(pattern, expand=False)

    if reference is not None:
        ref_mask = index.str.match(reference)
        filtered_ref = table.loc[ref_mask].copy()
        filtered_ref.index = index[ref_mask].str.extract(reference, expand=False)
        filtered = filtered.divide(filtered_ref * reference_multiplier)
        filtered = filtered.dropna(axis=0, how="all")

    if str(select_value).lower() == "avg":
        return filtered.mean(axis=1)

    column = str(select_value)
    if column not in filtered.columns:
        raise KeyError("Column '{}' not found in indicator table.".format(column))

    return filtered[column]


def build_indicator_summary(
    output_bundle,
    pattern,
    select_value,
    reference=None,
    reference_multiplier=1e3,
):
    """Build a metric x scenario table from output.csv files."""
    result = {}

    for scenario, table in output_bundle.items():
        result[scenario] = extract_indicator_values(
            table=table,
            pattern=pattern,
            select_value=select_value,
            reference=reference,
            reference_multiplier=reference_multiplier,
        )

    final = pd.DataFrame(result)
    final.index.name = "Metric"
    final.columns.name = "Scenario"
    return final


def _flatten_axes(axes):
    if isinstance(axes, np.ndarray):
        return list(axes.flatten())
    return [axes]


def plot_indicator_bar_grid(summary, same_axis_limits, label_size):
    """Plot one bar chart per indicator row."""
    if summary.empty:
        raise ValueError("Summary dataframe is empty; no bar chart to plot.")

    panel_count = len(summary.index)
    n_cols = min(3, max(1, panel_count))
    n_rows = math.ceil(panel_count / n_cols)

    axes = summary.T.plot.bar(
        rot=0,
        subplots=True,
        layout=(n_rows, n_cols),
        figsize=(5.2 * n_cols, 4.2 * n_rows),
        legend=False,
    )

    flat_axes = _flatten_axes(axes)

    if same_axis_limits:
        values = summary.values.flatten()
        ymin = float(np.nanmin(values))
        ymax = float(np.nanmax(values))
        margin = (ymax - ymin) * 0.05 if ymax > ymin else 0.05
        for ax in flat_axes:
            ax.set_ylim(ymin - margin, ymax + margin)

    for ax in flat_axes:
        if not ax.has_data():
            ax.set_visible(False)
            continue
        ax.title.set_fontsize(label_size)
        ax.xaxis.label.set_fontsize(label_size)
        ax.yaxis.label.set_fontsize(label_size)
        ax.tick_params(axis="both", labelsize=label_size)

    plt.tight_layout()
    plt.show()


def plot_subsidies_gap_boxplot(bundle, output_file):
    """Draw and save scenario-wise boxplots for Subsidies Gap."""
    data = pd.concat(bundle, names=["Scenario"]).reset_index(level=0)
    if "Subsidies Gap" not in data.columns:
        raise KeyError("`Subsidies Gap` column not found in distortion tables.")

    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(12.8, 9.6))
    box = sns.boxplot(data=data, x="Scenario", y="Subsidies Gap", ax=ax)

    medians = data.groupby("Scenario")["Subsidies Gap"].median()
    offset = data["Subsidies Gap"].median() * 0.05

    for tick, label in enumerate(box.get_xticklabels()):
        scenario = label.get_text()
        if scenario in medians.index:
            value = medians.loc[scenario]
            box.text(
                tick,
                value + offset,
                "{:.0f}".format(value),
                ha="center",
                size="x-small",
                color="w",
                weight="semibold",
            )

    ax.set_xlabel("")
    ax.set_ylabel("Subsidies Gap")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.savefig(output_file)
    plt.show()


def plot_subsidy_vs_distortion(
    bundle,
    output_folder,
    technology_filter,
):
    """Save one scatter plot per scenario using shared axis limits."""
    xmin = min(df["Subsidies"].min() for df in bundle.values())
    xmax = max(df["Subsidies"].max() for df in bundle.values())
    ymin = min(df["Distortion"].min() for df in bundle.values())
    ymax = max(df["Distortion"].max() for df in bundle.values())

    for scenario, table in bundle.items():
        plot_df = table.reset_index(drop=True)

        if "Technology" in plot_df.columns:
            plot_df = plot_df[~plot_df["Technology"].isin(technology_filter)]

        if plot_df.empty:
            continue

        make_scatter_plot(
            plot_df,
            "Subsidies",
            "Distortion",
            "Subsidies (Thousand euro)",
            "Distortion (Thousand euro)",
            annotate=False,
            save=output_folder / "subsidies_distortion_{}.png".format(scenario),
            format_y=lambda y, _: "{:.0f}".format(y / 1e3),
            format_x=lambda x, _: "{:.0f}".format(x / 1e3),
            s=10,
            diagonal_line=True,
            col_colors=None,
            xmin=xmin,
            ymin=ymin,
            xmax=xmax,
            ymax=ymax,
        )


In [ ]:
run_folder = resolve_run("policy_assessment", str(Path("subsidies_distortions") / run_name))
output_folder = run_folder / "img"
output_folder.mkdir(parents=True, exist_ok=True)

print(f"Run folder: {run_folder}")
print(f"Output folder: {output_folder}")

distortion_bundle = load_child_csv_bundle(run_folder, "subsidies_distortion.csv", index_col=None)
output_bundle = load_output_bundle(run_folder)

if not distortion_bundle:
    raise FileNotFoundError(f"No subsidies_distortion.csv files found in {run_folder}")
if not output_bundle:
    raise FileNotFoundError(f"No output.csv files found in {run_folder}")

print(f"Loaded distortion tables: {len(distortion_bundle)}")
print(f"Loaded output tables: {len(output_bundle)}")


In [ ]:
distortion_bundle = keep_scenarios(distortion_bundle, selected_scenarios)
output_bundle = keep_scenarios(output_bundle, selected_scenarios)

if selected_scenarios:
    print(f"Filtered scenarios: {list(distortion_bundle.keys())}")

if not distortion_bundle or not output_bundle:
    raise ValueError("Scenario filter removed all data. Update `selected_scenarios`.")

distortion_bundle = add_subsidies_gap(distortion_bundle)


In [ ]:
final_df = build_indicator_summary(
    output_bundle=output_bundle,
    pattern=indicator_pattern,
    select_value=select,
    reference=reference_pattern,
    reference_multiplier=reference_scale,
)

if final_df.empty:
    raise ValueError("No indicators matched the configured regex pattern.")

summary_path = output_folder / f"subsidies_{select}.csv"
final_df.to_csv(summary_path, index=True)
print(f"Saved summary: {summary_path}")

final_df


In [ ]:
plot_indicator_bar_grid(final_df, same_axis_limits=same_ylim, label_size=fontsize)


In [ ]:
plot_subsidies_gap_boxplot(distortion_bundle, output_folder / "subsidies_gap.png")


In [ ]:
plot_subsidy_vs_distortion(
    bundle=distortion_bundle,
    output_folder=output_folder,
    technology_filter=remove_technologies,
)
